# ML Workshop


## Identifying Loan Defaulters using XGBoost Classifier leveraging Snowpark ML

# Importing Libraries


## Importing standard libraries and specific column related functions

In [ ]:
!pip install numpy==1.24.3

In [3]:
# Importing necessary Libraries
import warnings
import pandas as pd
import numpy as np
from snowflake.ml.modeling.impute import SimpleImputer
from snowflake.ml.modeling.metrics import accuracy_score
from snowflake.ml.modeling.model_selection import GridSearchCV
from snowflake.ml.modeling.model_selection import RandomizedSearchCV
from snowflake.ml.modeling.preprocessing import LabelEncoder
from snowflake.snowpark import Session
from snowflake.snowpark import types as T
from snowflake.snowpark.functions import col
import snowflake.ml.modeling.preprocessing as snowml
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.metrics.correlation import correlation
import joblib
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.ml.registry import Registry
from snowflake.snowpark.functions import col, when
from snowflake.snowpark.functions import col, round as snow_round


warnings.simplefilter(action="ignore", category=UserWarning)

## Creating an Active Session

Connect your Python code to the Snowflake environment — specifically to the currently running Snowflake session that's managing your compute resources, access to data, and other execution context.

In [6]:
#Create a session
session = get_active_session()

#### Retrieve data from the database

In [7]:
#Data retrieval
loan_df = session.table("PUBLIC.LOAN_DEFAULT")
loan_df.show()

----------------------------------------------------------------------------------------------------------------------------------------
|"NAME"   |"MARITAL_STATUS"  |"DAYS_ACT_OPEN"  |"AGE"  |"INCOME"  |"ANY_PREVIOUS_DEFAULT"  |"GENDER"  |"OCCUPATION"  |"LOAN_APPROVAL"  |
----------------------------------------------------------------------------------------------------------------------------------------
|John     |Single            |2058             |28     |3000      |False                   |M         |Engineer      |1                |
|Mary     |Married           |2487             |34     |5000      |True                    |F         |Teacher       |0                |
|David    |Single            |1577             |22     |2000      |False                   |M         |Doctor        |1                |
|Sarah    |Married           |2896             |40     |8000      |False                   |F         |Engineer      |1                |
|Mike     |Single            |2349       

## Feature Engineering

We derive two new features 

Feature 1: INCOME_PER_MONTH_OF_LOAN - it is calculated based on existing income-related fields, normalized to a monthly basis to ensure consistency and comparability across customers with different income frequencies.

### INCOME_PER_MONTH_OF_LOAN = (Monthly Income X Loan Term) / Loan Amount

Helps assess affordability. It is a derived feature that helps evaluate a borrower's ability to repay their loan. It calculates how much total income the borrower earns during the loan term relative to the loan amount—but represented per month of the loan.

Feature 2: CREDIT_SCORE_BAND - Based on the credit score, we bucket them into five different categories 

#### Credit_score_band: Poor (<580), Fair (580–669), Good (670–739), Very Good (740–799), Excellent (800+)

In [ ]:
loan_df = loan_df.with_column(
    "INCOME_PER_MONTH_OF_LOAN",
    snow_round((col("MONTHLY_INCOME") * col("LOAN_TERM_MONTHS")) / col("LOAN_AMOUNT"), 2)
)


In [ ]:
loan_df = loan_df.with_column(
    "CREDIT_SCORE_BAND",
    when(col("CREDIT_SCORE") < 580, "Poor")
    .when((col("CREDIT_SCORE") >= 580) & (col("CREDIT_SCORE") <= 669), "Fair")
    .when((col("CREDIT_SCORE") >= 670) & (col("CREDIT_SCORE") <= 739), "Good")
    .when((col("CREDIT_SCORE") >= 740) & (col("CREDIT_SCORE") <= 799), "Very Good")
    .otherwise("Excellent")
)

## Seperate Categorical and Numerical Columns

### Categorical Columns: 
Columns representing discrete groups or labels, such as color, gender, or product type.

### Continuous Columns: 
Columns with numerical values that can take any range, like age, height, or income.

In [10]:
# Seperating Categorical and continuous variables
cat_cols = ['EMPLOYMENT_STATUS', 'EDUCATION_LEVEL', 'LOAN_PURPOSE', 'STATE','CREDIT_SCORE_BAND']
cont_cols = ['LOAN_AMOUNT', 'CREDIT_SCORE', 'MONTHLY_INCOME','DEBT_TO_INCOME_RATIO','LOAN_TERM_MONTHS','INCOME_PER_MONTH_OF_LOAN']

### Define Output Columns for Label Encoding

Defining output columns helps store encoded labels clearly, ensuring consistent mapping and easy reference.

In [12]:
# Define output columns for label encoding
output_cat_cols = ['EMPLOYMENT_STATUS_LE', 'EDUCATION_LEVEL_LE', 'LOAN_PURPOSE_LE', 'STATE_LE','CREDIT_SCORE_BAND_LE']


----------------------------------------------------------------------------------------------------------------------------------------
|"OCCUPATION"  |"ANY_PREVIOUS_DEFAULT"  |"MARITAL_STATUS"  |"GENDER"  |"NAME"   |"DAYS_ACT_OPEN"  |"AGE"  |"INCOME"  |"LOAN_APPROVAL"  |
----------------------------------------------------------------------------------------------------------------------------------------
|2.0           |0.0                     |1.0               |1.0       |John     |2058             |28     |3000      |1                |
|5.0           |1.0                     |0.0               |0.0       |Mary     |2487             |34     |5000      |0                |
|1.0           |0.0                     |1.0               |1.0       |David    |1577             |22     |2000      |1                |
|2.0           |0.0                     |0.0               |0.0       |Sarah    |2896             |40     |8000      |1                |
|0.0           |1.0                     |

## Label Encoding

Encoding the categorical column and the encoded values are stored in the "output_cat_cols", with the help of Pipeline preprocessor function and viewing the outputs.

### Pipeline: 
The pipeline consists of pre-processing steps. In this case, label encoding is the only preprocessing involved.

In [ ]:
#Creating a pipeline
# Create a list of label encoding steps for each categorical column, 
pipeline_steps = [
    (f"{input_col}_LE", LabelEncoder(input_cols=[input_col], output_cols=[output_col]))
    for input_col, output_col in zip(cat_cols, output_cat_cols)
]

# Define the preprocessing pipeline
preprocessing_pipeline = Pipeline(steps=pipeline_steps)
#preprocessing_artifact

# Fit and transform the DataFrame
transformed_df = preprocessing_pipeline.fit(loan_df).transform(loan_df)

# View column names
print(np.array(transformed_df.columns))

## Saving the preprocessor

Saving the preprocessor pipeline in a joblib format. This would later be invoked for running predictions.

In [ ]:
# Save the pipeline locally (if needed)
PIPELINE_FILE = 'preprocessing_pipeline.joblib'
joblib.dump(preprocessing_pipeline, PIPELINE_FILE)

#### Push the .joblib file to the respective stage, so that it can be re-used for predictions.


In [ ]:
# pushing the pickle file to stage
session.file.put(PIPELINE_FILE, "@PUBLIC.ML_WORKSHOP_STG", overwrite=True)

Dropping the original categorical columns, since the categories are encoded and stored

In [13]:
# Dropping the original categorical variables before Modelling 
transformed_df= transformed_df.drop("STATE","LOAN_PURPOSE","EDUCATION_LEVEL","EMPLOYMENT_STATUS","CREDIT_SCORE_BAND")

Split data into training and testing sets for model evaluation.

In [14]:
# Splitting of Data
train_df, test_df = transformed_df.random_split(weights=[0.8, 0.2], seed=8)
test_df1=test_df
# test_df=test_df.drop("LOAN_DEFAULT")

Build an initial model using XGB Classifier

In [15]:
# Creating a XGBClassifier Model
Classifier = XGBClassifier(
    input_cols=train_df.drop("LOAN_DEFAULT").columns,
    label_cols="LOAN_DEFAULT",
    output_cols="PRED_DEFAULT"
)

### Training the Model

In [16]:
# Fitting of data
Classifier.fit(train_df)

The version of package 'xgboost' in the local environment is 1.7.5, which does not fit the criteria for the requirement 'xgboost==1.7.3'. Your UDF might not work when the package version is different between the server and your local environment.
The version of package 'lightgbm' in the local environment is 3.3.3, which does not fit the criteria for the requirement 'lightgbm==3.3.5'. Your UDF might not work when the package version is different between the server and your local environment.


### Using the trained model to predict the test data

In [ ]:
#Prediction on test data
result=Classifier.predict(test_df)
result.show()



### Initializing registry

In [17]:
# Create a registry and log the model
# You can specify a different DB and Schema if you'd like otherwise it uses the session context
reg = Registry(session=session, database_name='DEV', schema_name='PUBLIC' )

## Model Logging

Logging our model into snowflake registry

In [ ]:
# Logging our model in the Registry

# Define model name and version (use uppercase for name)
model_name = "LOAN"
model_version = 'V0'

# Get sample input data to pass into the registry logging function
X = train_df.drop("LOAN_DEFAULT")

# Let's first log the very first model we trained
model_ver = reg.log_model(
    model_name=model_name,
    version_name=model_version,
    model=Classifier,
    sample_input_data=X, # to provide the feature schema
)

# # # Add evaluation metric for our model in registry
# model_ver.set_metric(
#     metric_name="accuracy",
#     value=accuracy,
# )

## Hyper-Parameter Tuning

Hyperparameter tuning is the process of optimizing model parameters that aren't learned from data to improve performance. Here we use RandomizedSearch CV.


## RandomizedSearch CV
RandomizedSearchCV is a hyperparameter tuning technique that randomly samples parameter combinations from a specified distribution and evaluates them using cross-validation to find the best model.

In [ ]:
# RandomizedSearchCV to get best model
random_search = RandomizedSearchCV(
    estimator=XGBClassifier(),
    param_distributions={},  # No manual parameters
    n_iter=10,               # Still needs a number of iterations
    n_jobs=-1,
    scoring="accuracy",
    input_cols=train_df.drop("LOAN_DEFAULT").columns,
    label_cols="LOAN_DEFAULT",
    output_cols="PRED_APPROVED",
)



In [ ]:
#Training the Tuned Model
random_search.fit(train_df)

In [22]:
# Retrieving Optimal model
optimal_model = random_search.to_sklearn().best_estimator_
optimal_model

RandomForestClassifier(max_depth=3, n_estimators=200, n_jobs=3)

Check the accuracy of the predictions

In [27]:
# Accuracy of model
accuracy = accuracy_score(
    df=result, y_true_col_names="LOAN_DEFAULT", y_pred_col_names="PRED_DEFAULT"
)

print(f"Accuracy: {accuracy}")

Accuracy: 1.0


# Model Registry

### Registering the latest optimal model to the registry, with a new version.

In [28]:
#Logging our best model in the Registry

#Define model name and version (use uppercase for name)
model_name = "LOAN"
model_version = 'V1'

# Let's first log the very first model we trained
model_ver = reg.log_model(
    model_name=model_name,
    version_name=model_version,
    model=optimal_model,
    sample_input_data=X, # to provide the feature schema
)

# # Add evaluation metric for our model in registry
model_ver.set_metric(
    metric_name="accuracy",
    value=accuracy,
)

In [29]:
# Viewing the models in our registry
reg.show_models()

,created_on,name,database_name,schema_name,comment,owner,default_version_name,versions
0,2024-03-06 00:02:15.909000-08:00,LOAN,LOAN,DEPARTMENT,None,ACCOUNTADMIN,V0,"[""V0"",""V1""]"


Setting the default model version

In [32]:
# Retrieving optimal model and making it as default model version for our model.
m = reg.get_model(model_name)
m.default = 'V1'
mod_v = m.default


Running predictions using the default model

In [39]:
# Predict the results 
run_prediction = mod_v.run(test_df, function_name="predict")
run_prediction=run_prediction.rename({'"output_feature_0"':'PRED_LOAN_DEFAULT'})
run_prediction.show()

---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"LOAN_APPROVAL"  |"OCCUPATION"  |"ANY_PREVIOUS_DEFAULT"  |"MARITAL_STATUS"  |"GENDER"  |"DAYS_ACT_OPEN"  |"AGE"  |"INCOME"  |"PREDICT_PROBA_0"     |"PREDICT_PROBA_1"    |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|1                |4.0           |0.0                     |0.0               |0.0       |2118             |29     |3500      |0.05133735708735709   |0.9486626429126428   |
|0                |2.0           |1.0                     |1.0               |1.0       |3400             |47     |6000      |0.8580779220779223    |0.14192207792207795  |
|1                |3.0           |0.0                     |0.0               |0.0       |2314             |32     |4500      |0.069713439338

Save the predicted results to a table

In [39]:
# To test in SQL write test data back to a table
test_df.write.mode("overwrite").save_as_table("TEST_DATA")

In [40]:
session.close()